In [ ]:
import os
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    drive = None
    IN_COLAB = False

DEFAULT_REPO_ROOT = Path(os.environ.get("CHESS_REPO_ROOT", "/content/drive/MyDrive/chess_engine"))
if not IN_COLAB:
    raise RuntimeError(
        "This notebook is intended for Google Colab. "
        "Open it in Colab and keep the repo under Google Drive."
    )

drive.mount("/content/drive", force_remount=False)
if not DEFAULT_REPO_ROOT.exists():
    raise FileNotFoundError(
        f"Missing repo root: {DEFAULT_REPO_ROOT}. "
        "Set CHESS_REPO_ROOT or move the repo to this Drive path."
    )
os.chdir(DEFAULT_REPO_ROOT)

print("IN_COLAB:", IN_COLAB)
print("cwd:", Path.cwd().resolve())

In [ ]:
import os
import sys
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "model").exists() and (candidate / "runs").exists() and (candidate / "train_v3_FT2").exists():
            return candidate
    raise RuntimeError(f"Cannot find repo root from: {start}")


REPO_ROOT = find_repo_root(Path.cwd())
RUNS_ROOT = Path(os.environ.get("CHESS_RUNS_ROOT", str(REPO_ROOT / "runs")))
MODEL_ROOT = REPO_ROOT / "model"
TRAIN_V3_DIR = REPO_ROOT / "train_v3_FT2"

SEED = 123
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

torch.set_float32_matmul_precision("high")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "cpu"
GPU_TOTAL_MEM_GB = float(torch.cuda.get_device_properties(0).total_memory / 1024**3) if DEVICE.type == "cuda" else 0.0

print(f"REPO_ROOT = {REPO_ROOT}")
print(f"RUNS_ROOT = {RUNS_ROOT}")
print(f"TRAIN_V3_DIR = {TRAIN_V3_DIR}")
print(f"DEVICE = {DEVICE}")
print(f"GPU_NAME = {GPU_NAME}")
print(f"GPU_TOTAL_MEM_GB = {GPU_TOTAL_MEM_GB:.2f}")

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import numpy as np


def env_bool(name: str, default: bool) -> bool:
    raw = os.environ.get(name, "").strip().lower()
    if not raw:
        return bool(default)
    return raw in {"1", "true", "yes", "y"}


DATA_ROOT_OVERRIDE = os.environ.get("CHESS_DATA_ROOT", "").strip()
DATA_ROOT = Path(DATA_ROOT_OVERRIDE) if DATA_ROOT_OVERRIDE else (REPO_ROOT / "data" / "process")
STAGE_DATA_LOCAL = IN_COLAB and env_bool("CHESS_STAGE_DATA_LOCAL", True)
FORCE_RESTAGE = env_bool("CHESS_FORCE_RESTAGE", False)
LOCAL_DATA_ROOT = Path(os.environ.get("CHESS_LOCAL_DATA_ROOT", "/content/chess_engine_data/process"))

if not DATA_ROOT.exists():
    raise FileNotFoundError(
        f"Missing data root: {DATA_ROOT}. "
        "Set CHESS_DATA_ROOT or place the shards under REPO_ROOT/data/process."
    )

DATA_ROOT_ACTIVE = DATA_ROOT
if STAGE_DATA_LOCAL:
    if FORCE_RESTAGE and LOCAL_DATA_ROOT.exists():
        shutil.rmtree(LOCAL_DATA_ROOT)
    local_ready = all((LOCAL_DATA_ROOT / split).exists() for split in ("train", "val", "test"))
    if not local_ready:
        LOCAL_DATA_ROOT.parent.mkdir(parents=True, exist_ok=True)
        src = str(DATA_ROOT).rstrip("/\\") + "/"
        dst = str(LOCAL_DATA_ROOT).rstrip("/\\") + "/"
        if shutil.which("rsync"):
            print(f"Staging dataset to local runtime disk via rsync: {DATA_ROOT} -> {LOCAL_DATA_ROOT}")
            subprocess.run(["rsync", "-a", "--delete", src, dst], check=True)
        else:
            print(f"Staging dataset to local runtime disk via shutil: {DATA_ROOT} -> {LOCAL_DATA_ROOT}")
            shutil.copytree(DATA_ROOT, LOCAL_DATA_ROOT, dirs_exist_ok=True)
    DATA_ROOT_ACTIVE = LOCAL_DATA_ROOT


def _sorted_npy_files(split_dir: Path, pattern: str):
    return sorted(split_dir.glob(pattern), key=lambda p: p.name)


def scan_split(split: str):
    split_dir = DATA_ROOT_ACTIVE / split
    if not split_dir.exists():
        raise FileNotFoundError(f"Missing split dir: {split_dir}")
    x_files = _sorted_npy_files(split_dir, "X_*.npy")
    y_files = _sorted_npy_files(split_dir, "y_*.npy")
    if len(x_files) == 0 or len(y_files) == 0:
        raise FileNotFoundError(f"No shards found in {split_dir}. Expect X_*.npy and y_*.npy")
    if len(x_files) != len(y_files):
        raise ValueError(f"Shard count mismatch in {split}: X={len(x_files)} vs y={len(y_files)}")

    shard_sizes = []
    for xf, yf in zip(x_files, y_files):
        X = np.load(xf, mmap_mode="r")
        y = np.load(yf, mmap_mode="r")
        if X.ndim != 4 or X.shape[1:] != (18, 8, 8):
            raise ValueError(f"Bad X shape at {xf.name}: {X.shape} (expect (N,18,8,8))")
        if y.ndim != 1:
            raise ValueError(f"Bad y shape at {yf.name}: {y.shape} (expect (N,))")
        if X.shape[0] != y.shape[0]:
            raise ValueError(f"N mismatch at {xf.name} vs {yf.name}: {X.shape[0]} vs {y.shape[0]}")
        shard_sizes.append(int(X.shape[0]))

    return {
        "split": split,
        "num_shards": len(x_files),
        "num_samples": int(sum(shard_sizes)),
        "first_shard": x_files[0].name,
        "last_shard": x_files[-1].name,
        "min_shard_size": int(min(shard_sizes)),
        "max_shard_size": int(max(shard_sizes)),
    }

split_rows = [scan_split(split) for split in ("train", "val", "test")]
print(f"DATA_ROOT = {DATA_ROOT}")
print(f"DATA_ROOT_ACTIVE = {DATA_ROOT_ACTIVE}")
print(f"STAGE_DATA_LOCAL = {STAGE_DATA_LOCAL}")
display(pd.DataFrame(split_rows))

In [ ]:
import importlib.util
import json
import os
import sys
from pathlib import Path


def resolve_train_v3_helper_path(repo_root: Path) -> Path:
    candidates = []
    env_dir = os.environ.get("CHESS_TRAIN_V3_DIR", "").strip()
    if env_dir:
        candidates.append(repo_root / env_dir / "ft2_colab_helpers.py")
    candidates.append(repo_root / "train_v3_FT2" / "ft2_colab_helpers.py")
    seen = set()
    for path in candidates:
        key = str(path)
        if key in seen:
            continue
        seen.add(key)
        if path.exists():
            return path
    raise FileNotFoundError(f"Cannot find ft2_colab_helpers.py under {repo_root}. Tried: {candidates}")


def load_ft2_helper(repo_root: Path):
    helper_name = "ft2_colab_helpers"
    helper_path = resolve_train_v3_helper_path(repo_root)
    if helper_name in sys.modules:
        del sys.modules[helper_name]
    spec = importlib.util.spec_from_file_location(helper_name, helper_path)
    lab = importlib.util.module_from_spec(spec)
    sys.modules[helper_name] = lab
    assert spec.loader is not None
    spec.loader.exec_module(lab)
    return lab, helper_path


def env_int(name: str, default: int) -> int:
    raw = os.environ.get(name, "").strip()
    return int(raw) if raw else int(default)


def env_optional_int(name: str, default):
    raw = os.environ.get(name, "").strip()
    return int(raw) if raw else default


def env_float(name: str, default: float) -> float:
    raw = os.environ.get(name, "").strip()
    return float(raw) if raw else float(default)


def env_bool(name: str, default: bool) -> bool:
    raw = os.environ.get(name, "").strip().lower()
    if not raw:
        return bool(default)
    return raw in {"1", "true", "yes", "y"}


lab, HELPER_PATH = load_ft2_helper(REPO_ROOT)
BASE_TRAIN_CFG = lab.FT2TrainConfig()

MODEL_CFG = {
    "num_blocks": 20,
    "hidden_dim": 256,
    "input_channels": 18,
    "drop_path_rate": 0.05,
    "output_mode": "tanh",
}

RUN_NAME = os.environ.get("CHESS_RUN_NAME", "dgrn_5m_ft2_t4_run1")
RUN_DIR = RUNS_ROOT / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
RESUME_IF_EXISTS = env_bool("CHESS_RESUME_IF_EXISTS", False)
LATEST_CKPT = RUN_DIR / "checkpoints" / "ckpt_latest.pt"
RUN_CONFIG_PATH = RUN_DIR / "reports" / "run_config.json"

saved_train_cfg = {}
RESUME_PROFILE_LOADED = False
if RESUME_IF_EXISTS and LATEST_CKPT.exists() and RUN_CONFIG_PATH.exists():
    saved_run_config = json.loads(RUN_CONFIG_PATH.read_text(encoding="utf-8"))
    saved_train_cfg = dict(saved_run_config.get("train_cfg", {}))
    RESUME_PROFILE_LOADED = True
    print("[resume-config] Reusing saved FT2 train profile from run_config.json")


def saved_or_default(name: str, fallback):
    return saved_train_cfg.get(name, fallback)


AUTOTUNE_PROFILE = (
    DEVICE.type == "cuda"
    and (not env_bool("CHESS_DISABLE_PROFILE_AUTOTUNE", False))
    and (not RESUME_PROFILE_LOADED)
)

TRAIN_CFG = lab.FT2TrainConfig(
    run_name=RUN_NAME,
    epochs=env_int("CHESS_EPOCHS", int(saved_or_default("epochs", BASE_TRAIN_CFG.epochs))),
    main_batch_size=env_int("CHESS_MAIN_BATCH_SIZE", int(saved_or_default("main_batch_size", BASE_TRAIN_CFG.main_batch_size))),
    clean_center_batch_size=env_int("CHESS_CLEAN_CENTER_BATCH_SIZE", int(saved_or_default("clean_center_batch_size", BASE_TRAIN_CFG.clean_center_batch_size))),
    ambiguous_center_batch_size=env_int("CHESS_AMBIGUOUS_CENTER_BATCH_SIZE", int(saved_or_default("ambiguous_center_batch_size", BASE_TRAIN_CFG.ambiguous_center_batch_size))),
    grad_accum_steps=env_int("CHESS_GRAD_ACCUM_STEPS", int(saved_or_default("grad_accum_steps", BASE_TRAIN_CFG.grad_accum_steps))),
    eval_batch_size=env_int("CHESS_EVAL_BATCH_SIZE", int(saved_or_default("eval_batch_size", BASE_TRAIN_CFG.eval_batch_size))),
    learning_rate=env_float("CHESS_LEARNING_RATE", float(saved_or_default("learning_rate", BASE_TRAIN_CFG.learning_rate))),
    min_lr=env_float("CHESS_MIN_LR", float(saved_or_default("min_lr", BASE_TRAIN_CFG.min_lr))),
    weight_decay=env_float("CHESS_WEIGHT_DECAY", float(saved_or_default("weight_decay", BASE_TRAIN_CFG.weight_decay))),
    grad_clip_norm=env_float("CHESS_GRAD_CLIP_NORM", float(saved_or_default("grad_clip_norm", BASE_TRAIN_CFG.grad_clip_norm))),
    seed=env_int("CHESS_SEED", int(saved_or_default("seed", BASE_TRAIN_CFG.seed))),
    train_num_shards=env_optional_int("CHESS_TRAIN_NUM_SHARDS", saved_or_default("train_num_shards", BASE_TRAIN_CFG.train_num_shards)),
    val_num_shards=env_int("CHESS_VAL_NUM_SHARDS", int(saved_or_default("val_num_shards", BASE_TRAIN_CFG.val_num_shards))),
    test_num_shards=env_int("CHESS_TEST_NUM_SHARDS", int(saved_or_default("test_num_shards", BASE_TRAIN_CFG.test_num_shards))),
    val_max_samples=env_int("CHESS_VAL_MAX_SAMPLES", int(saved_or_default("val_max_samples", BASE_TRAIN_CFG.val_max_samples))),
    test_max_samples=env_int("CHESS_TEST_MAX_SAMPLES", int(saved_or_default("test_max_samples", BASE_TRAIN_CFG.test_max_samples))),
    log_every_steps=env_int("CHESS_LOG_EVERY_STEPS", int(saved_or_default("log_every_steps", BASE_TRAIN_CFG.log_every_steps))),
    grad_monitor_every_steps=env_int("CHESS_GRAD_MONITOR_EVERY_STEPS", int(saved_or_default("grad_monitor_every_steps", BASE_TRAIN_CFG.grad_monitor_every_steps))),
    use_amp=env_bool("CHESS_USE_AMP", bool(saved_or_default("use_amp", BASE_TRAIN_CFG.use_amp))),
    amp_dtype=os.environ.get("CHESS_AMP_DTYPE", str(saved_or_default("amp_dtype", BASE_TRAIN_CFG.amp_dtype))).strip() or BASE_TRAIN_CFG.amp_dtype,
    amp_loss_scale=env_float("CHESS_AMP_LOSS_SCALE", float(saved_or_default("amp_loss_scale", BASE_TRAIN_CFG.amp_loss_scale))),
    preload_shard_dtype=os.environ.get("CHESS_PRELOAD_SHARD_DTYPE", str(saved_or_default("preload_shard_dtype", BASE_TRAIN_CFG.preload_shard_dtype))).strip() or str(BASE_TRAIN_CFG.preload_shard_dtype),
    channels_last=env_bool("CHESS_CHANNELS_LAST", bool(saved_or_default("channels_last", BASE_TRAIN_CFG.channels_last))),
    cudnn_benchmark=env_bool("CHESS_CUDNN_BENCHMARK", bool(saved_or_default("cudnn_benchmark", BASE_TRAIN_CFG.cudnn_benchmark))),
    pin_memory_batches=env_bool("CHESS_PIN_MEMORY_BATCHES", bool(saved_or_default("pin_memory_batches", BASE_TRAIN_CFG.pin_memory_batches))),
    prefetch_shards=env_bool("CHESS_PREFETCH_SHARDS", bool(saved_or_default("prefetch_shards", BASE_TRAIN_CFG.prefetch_shards))),
    prefetch_workers=env_int("CHESS_PREFETCH_WORKERS", int(saved_or_default("prefetch_workers", BASE_TRAIN_CFG.prefetch_workers))),
    use_backbone_pcgrad=env_bool("CHESS_USE_BACKBONE_PCGRAD", bool(saved_or_default("use_backbone_pcgrad", BASE_TRAIN_CFG.use_backbone_pcgrad))),
    benchmark_steps=env_int("CHESS_BENCHMARK_STEPS", int(saved_or_default("benchmark_steps", BASE_TRAIN_CFG.benchmark_steps))),
    benchmark_warmup_steps=env_int("CHESS_BENCHMARK_WARMUP_STEPS", int(saved_or_default("benchmark_warmup_steps", BASE_TRAIN_CFG.benchmark_warmup_steps))),
    benchmark_num_shards=env_int("CHESS_BENCHMARK_NUM_SHARDS", int(saved_or_default("benchmark_num_shards", BASE_TRAIN_CFG.benchmark_num_shards))),
    max_profile_mem_ratio=env_float("CHESS_MAX_PROFILE_MEM_RATIO", float(saved_or_default("max_profile_mem_ratio", BASE_TRAIN_CFG.max_profile_mem_ratio))),
    periodic_save_minutes=env_int("CHESS_PERIODIC_SAVE_MINUTES", int(saved_or_default("periodic_save_minutes", 30))),
    save_epoch_checkpoints=env_bool("CHESS_SAVE_EPOCH_CHECKPOINTS", bool(saved_or_default("save_epoch_checkpoints", False))),
    resume_if_exists=RESUME_IF_EXISTS,
    role_val_frac=env_float("CHESS_ROLE_VAL_FRAC", float(saved_or_default("role_val_frac", BASE_TRAIN_CFG.role_val_frac))),
    role_split_seed=env_int("CHESS_ROLE_SPLIT_SEED", int(saved_or_default("role_split_seed", BASE_TRAIN_CFG.role_split_seed))),
    role_refresh_cache=env_bool("CHESS_ROLE_REFRESH_CACHE", bool(saved_or_default("role_refresh_cache", BASE_TRAIN_CFG.role_refresh_cache))),
)
GATE_CFG = lab.FT2GateConfig(
    midband_mae_rel_tol=env_float("CHESS_MIDBAND_MAE_REL_TOL", 0.05),
    stable_slope_abs_tol=env_float("CHESS_STABLE_SLOPE_ABS_TOL", 0.02),
)

print("FT2 helper:", HELPER_PATH)
print("RUN_DIR:", RUN_DIR)
print("DATA_ROOT_ACTIVE:", DATA_ROOT_ACTIVE)
print("AUTOTUNE_PROFILE:", AUTOTUNE_PROFILE)
print("TRAIN_CFG:", TRAIN_CFG)
print("GATE_CFG:", GATE_CFG)
print("periodic latest checkpoint every (minutes):", TRAIN_CFG.periodic_save_minutes)
print("save epoch checkpoints:", TRAIN_CFG.save_epoch_checkpoints)

In [ ]:
run_artifacts = lab.run_ft2_training(
    repo_root=REPO_ROOT,
    runs_root=RUNS_ROOT,
    data_root=DATA_ROOT_ACTIVE,
    model_cfg=MODEL_CFG,
    train_cfg=TRAIN_CFG,
    gate_cfg=GATE_CFG,
    autotune_profile=AUTOTUNE_PROFILE,
)

RUN_DIR = Path(run_artifacts["paths"]["run_dir"])
SELECTED_CHECKPOINT = Path(run_artifacts["selected_checkpoint"])
print("Run finished.")
print("Run dir:", RUN_DIR)
print("Selected checkpoint:", SELECTED_CHECKPOINT)
print("Final center_score:", run_artifacts["final_eval"].get("center_score"))
print("Final oracle_midband_mae_sum_stable:", run_artifacts["final_eval"].get("oracle_midband_mae_sum_stable"))
print("Final oracle_stable_0.7_slope:", run_artifacts["final_eval"].get("oracle_stable_0.7_slope"))

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

reports_dir = RUN_DIR / "reports"
history_df = pd.read_csv(reports_dir / "history.csv")
step_history_path = reports_dir / "step_history.csv"
step_hist = pd.read_csv(step_history_path) if step_history_path.exists() else pd.DataFrame()
decision = json.loads((reports_dir / "decision_summary.json").read_text(encoding="utf-8"))
l4_ref = json.loads((reports_dir / "l4_reference.json").read_text(encoding="utf-8"))
selected_eval = json.loads((reports_dir / "selected_checkpoint_eval.json").read_text(encoding="utf-8"))

print("Decision summary:", decision)
display(history_df.tail(5))

compare_keys = [
    "oracle_midband_mae_sum_stable",
    "oracle_stable_0.7_slope",
    "oracle_center_amp_ratio",
    "oracle_center_false_0.1eq",
    "oracle_center_false_0.2eq",
    "pooled_center_mae",
    "pooled_center_amp_ratio",
    "pooled_center_false_0.1eq",
    "pooled_center_false_0.2eq",
    "center_score",
    "clean_center_mae",
    "clean_center_amp_ratio",
    "clean_center_false_0.1eq",
    "ambiguous_center_mae",
    "ambiguous_center_amp_ratio",
    "ambiguous_center_false_0.1eq",
]
compare = pd.DataFrame([
    {"label": "L4_reference", **{k: l4_ref.get(k, l4_ref.get("primary", {}).get(k)) for k in compare_keys}},
    {"label": "FT2_selected", **{k: selected_eval.get(k) for k in compare_keys}},
])
display(compare.round(6))

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

axes[0, 0].plot(history_df["epoch"], history_df["train_main_objective"], label="train_main_objective")
axes[0, 0].plot(history_df["epoch"], history_df["train_clean_fit"], label="train_clean_fit")
axes[0, 0].plot(history_df["epoch"], history_df["train_ambiguous"], label="train_ambiguous")
axes[0, 0].set_title("FT2 Train Objectives")
axes[0, 0].set_xlabel("epoch")
axes[0, 0].set_ylabel("objective")
axes[0, 0].legend()

axes[0, 1].plot(history_df["epoch"], history_df["oracle_midband_mae_sum_stable"], marker="o", label="FT2 midband MAE")
axes[0, 1].axhline(float(l4_ref["primary"]["oracle_midband_mae_sum_stable"]), color="tab:gray", linestyle="--", label="L4 midband MAE")
ax2 = axes[0, 1].twinx()
ax2.plot(history_df["epoch"], history_df["oracle_stable_0.7_slope"], marker="s", color="tab:green", label="FT2 slope")
ax2.axhline(float(l4_ref["primary"]["oracle_stable_0.7_slope"]), color="tab:olive", linestyle="--", label="L4 slope")
axes[0, 1].set_title("Hard A-Gate Metrics")
axes[0, 1].set_xlabel("epoch")
axes[0, 1].set_ylabel("oracle_midband_mae_sum_stable")
ax2.set_ylabel("oracle_stable_0.7_slope")
lines1, labels1 = axes[0, 1].get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
axes[0, 1].legend(lines1 + lines2, labels1 + labels2, loc="best")

axes[1, 0].plot(history_df["epoch"], history_df["center_score"], marker="o", label="FT2 center_score")
axes[1, 0].axhline(float(l4_ref["primary"]["center_score"]), color="tab:gray", linestyle="--", label="L4 center_score")
axes[1, 0].set_title("Center Score vs L4")
axes[1, 0].set_xlabel("epoch")
axes[1, 0].set_ylabel("center_score")
axes[1, 0].legend()

if len(step_hist) > 0:
    axes[1, 1].plot(step_hist["global_step"], step_hist["grad_conflict_backbone"], alpha=0.8, label="conflict_pre")
    axes[1, 1].plot(step_hist["global_step"], step_hist["grad_conflict_backbone_post"], alpha=0.8, label="conflict_post")
    axes[1, 1].set_title("Gradient Conflict")
    axes[1, 1].set_xlabel("global_step")
    axes[1, 1].set_ylabel("mean negative cosine magnitude")
    axes[1, 1].legend(loc="best")
else:
    axes[1, 1].text(0.5, 0.5, "No step_history rows yet", ha="center", va="center")
    axes[1, 1].set_axis_off()

plt.tight_layout()
plt.show()